© 2026 Mechanome Bio. Proprietary and confidential.

# Set up SageMaker instance: environments + input data
Run from the repo root (the cloned repo is the working directory).

In [ ]:
from pathlib import Path

DATASET_NAME = "20240208_core_s0005_spt-xen_onco_pancreas"
S3_DATASETS = "s3://mechanome-cellpose-airr-lon-161012475647-eu-west-2/benchmark/_datasets"

## Input data
Zarr (spatialdata) store from S3 into `inputs/zarrs/`.

Also needed, copied in by hand:
- ground truths -> `inputs/ground_truths/<dataset>/` (`crop-N-x<X>-y<Y>-flu.tif` / `-mask.tif` pairs), for GeneSegNet
- BIDCell reference csvs -> `config_files/BIDCell_references/` (see BidCell.ipynb)

In [ ]:
s3_zarr = f"{S3_DATASETS}/{DATASET_NAME}/zarr/{DATASET_NAME}.zarr"
local_zarr = Path("inputs/zarrs") / f"{DATASET_NAME}.zarr"
!aws s3 sync {s3_zarr} {local_zarr}

## Conda environments
Three envs:
- `bidcell_segger_cellist` (shared, built here): BIDCell, Segger, Cellist. Also the kernel for GeneSegNet.ipynb and example_method_notebook.ipynb.
- `rna2seg`: built at the top of RNA2seg.ipynb.
- `genesegnet`: built at the top of GeneSegNet.ipynb (python 3.8, called with `conda run`).

Segger is installed with `--no-deps`: it pins numpy<2, but spatialdata>=0.6 (needed to read zarr v3) needs numpy>=2.
Its other dependencies are listed in the requirements file.

In [ ]:
!conda create -y -n bidcell_segger_cellist python=3.11 pip
!conda run --no-capture-output -n bidcell_segger_cellist pip install -r env_files/bidcell_segger_cellist.txt
!conda run --no-capture-output -n bidcell_segger_cellist pip install --no-deps "segger @ git+https://github.com/EliHei2/segger_dev.git@e7d5c7076214db41eceae704fd489ef245e0b7ac"
!conda run -n bidcell_segger_cellist python -m ipykernel install --user --name bidcell_segger_cellist --display-name "bidcell_segger_cellist"